In [25]:
# =============================================================================
# Phase 6 — Kafka Producer : Silver Featured → Kafka Stream
# Craigslist Used Vehicle Dataset
# =============================================================================

import logging
import time
import json
import subprocess
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
log = logging.getLogger("VehicleKafkaProducer")

In [26]:
# =============================================================================
# CONFIG
# =============================================================================

KAFKA_BROKER         = "localhost:9092"
KAFKA_TOPIC          = "vehicle_market_stream"
SILVER_FEATURED_PATH = "hdfs:///user/vehicle_market/silver/vehicles_featured"

# Make Kafka CLI executable configurable for portability across installations
# Common alternatives: "bin/kafka-topics.sh", "/usr/local/kafka/bin/kafka-topics.sh"
KAFKA_TOPICS_CMD     = "kafka-topics.sh"

# Publishing controls
RECORDS_TO_PUBLISH   = 10000   # None publishes the full Silver Featured dataset.
                                # Smaller values simulate limited real-time traffic.
BATCH_SIZE           = 100     # Records per Kafka write batch
PUBLISH_DELAY_SEC    = 0.5     # Pause between batches to simulate live arrival rate

# Columns to include in each Kafka message.
# Deliberately excludes: description, image_url, region_url, url, VIN
# — these are high-cardinality or free-text fields with no streaming value.
# Includes engineered temporal and analytical features: posting_period,
# is_weekend, depreciation_index — computed during Phase 3 Feature Engineering.
STREAM_COLUMNS = [
    "id", "manufacturer", "model", "year", "price", "odometer",
    "fuel", "transmission", "condition", "title_status", "drive",
    "type", "paint_color", "state", "lat", "long", "posting_date",
    "vehicle_age", "age_group", "price_category", "mileage_category",
    "price_per_mile", "posting_year", "posting_month", "posting_season",
    "is_luxury", "is_alternative_fuel", "vehicle_segment",
    "clean_title", "is_salvage", "is_automatic", "price_band",
    "posting_period", "is_weekend", "depreciation_index"
]

In [27]:
# =============================================================================
# 1. SPARK SESSION
# =============================================================================

spark = SparkSession.builder \
    .appName("VehicleMarket_KafkaProducer") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"
    ) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

pipeline_start = time.time()
log.info("="*65)
log.info("  Phase 6 — Kafka Producer : Silver Featured → Kafka")
log.info("="*65)



2026-07-26 00:42:10 | INFO | =================================================================
2026-07-26 00:42:10 | INFO |   Phase 6 — Kafka Producer : Silver Featured → Kafka
2026-07-26 00:42:10 | INFO | =================================================================


In [28]:
# =============================================================================
# 2. VERIFY KAFKA BROKER AND TOPIC AVAILABILITY
# =============================================================================

log.info(f"Verifying Kafka broker at: {KAFKA_BROKER}...")

def verify_kafka_broker(broker, cmd):
    """Check if Kafka broker is reachable using kafka-topics CLI.
    KAFKA_TOPICS_CMD is configurable to support different installations.
    """
    try:
        result = subprocess.run(
            [cmd, "--bootstrap-server", broker, "--list"],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            topics = [
                t for t in result.stdout.strip().split("\n") if t
            ]
            return True, topics
        return False, result.stderr.strip()
    except FileNotFoundError:
        return False, (
            f"'{cmd}' not found on PATH. "
            f"Set KAFKA_TOPICS_CMD to the full path of kafka-topics.sh."
        )
    except Exception as e:
        return False, str(e)

broker_ok, broker_info = verify_kafka_broker(KAFKA_BROKER, KAFKA_TOPICS_CMD)

if not broker_ok:
    raise ConnectionError(
        f"Kafka broker not reachable at {KAFKA_BROKER}.\n"
        f"Start Kafka before running this notebook.\n"
        f"Details: {broker_info}"
    )

existing_topics = broker_info if isinstance(broker_info, list) else []
log.info(f"  Broker reachable.")
log.info(f"  Existing topics: {existing_topics}")



2026-07-26 00:42:10 | INFO | Verifying Kafka broker at: localhost:9092...
2026-07-26 00:42:11 | INFO |   Broker reachable.
2026-07-26 00:42:11 | INFO |   Existing topics: ['vehicle_market_stream']


In [29]:
# =============================================================================
# 3. CREATE TOPIC IF NOT EXISTS
# =============================================================================

if KAFKA_TOPIC in existing_topics:
    topic_status = "existing"
    log.info(f"  Topic '{KAFKA_TOPIC}' — already exists, reusing.")
else:
    log.info(f"  Topic '{KAFKA_TOPIC}' — not found, creating...")
    create_result = subprocess.run(
        [
            KAFKA_TOPICS_CMD,
            "--bootstrap-server", KAFKA_BROKER,
            "--create",
            "--topic", KAFKA_TOPIC,
            "--partitions", "3",
            "--replication-factor", "1"
        ],
        capture_output=True, text=True, timeout=15
    )
    if create_result.returncode == 0:
        topic_status = "created"
        log.info(f"  Topic '{KAFKA_TOPIC}' — created successfully (3 partitions).")
    else:
        raise RuntimeError(
            f"Failed to create topic '{KAFKA_TOPIC}': "
            f"{create_result.stderr.strip()}"
        )


2026-07-26 00:42:11 | INFO |   Topic 'vehicle_market_stream' — already exists, reusing.


In [30]:
# =============================================================================
# 4. LOAD SILVER FEATURED DATASET
# =============================================================================

log.info("Loading Silver Featured dataset...")
df_full = spark.read.parquet(SILVER_FEATURED_PATH)
total_available = df_full.count()
log.info(f"  Total records available : {total_available}")
log.info(f"  Total columns           : {len(df_full.columns)}")



2026-07-26 00:42:11 | INFO | Loading Silver Featured dataset...
2026-07-26 00:42:12 | INFO |   Total records available : 242979
2026-07-26 00:42:12 | INFO |   Total columns           : 49


In [31]:
# =============================================================================
# 5. VALIDATE REQUIRED STREAM COLUMNS
# =============================================================================

log.info("Validating stream columns...")
missing_cols = [c for c in STREAM_COLUMNS if c not in df_full.columns]
if missing_cols:
    raise ValueError(
        f"Silver Featured is missing stream columns: {missing_cols}. "
        f"Re-run Phase 3 before proceeding."
    )
log.info(f"  All {len(STREAM_COLUMNS)} stream columns present.")


2026-07-26 00:42:12 | INFO | Validating stream columns...
2026-07-26 00:42:12 | INFO |   All 35 stream columns present.


In [32]:
# =============================================================================
# 6. PREPARE STREAM DATASET
# =============================================================================

log.info("Preparing stream dataset...")

df_stream = df_full.select(STREAM_COLUMNS)
df_stream = df_stream.orderBy(F.rand(seed=42))

if RECORDS_TO_PUBLISH is not None:
    df_stream = df_stream.limit(RECORDS_TO_PUBLISH)

records_to_send = df_stream.count()
log.info(f"  Records selected for streaming : {records_to_send}")
log.info(f"  Batch size                     : {BATCH_SIZE}")
log.info(f"  Delay between batches          : {PUBLISH_DELAY_SEC}s")
estimated_time = (records_to_send / BATCH_SIZE) * PUBLISH_DELAY_SEC
log.info(f"  Estimated publish time         : ~{estimated_time:.0f}s")




2026-07-26 00:42:12 | INFO | Preparing stream dataset...
2026-07-26 00:42:12 | INFO |   Records selected for streaming : 10000
2026-07-26 00:42:12 | INFO |   Batch size                     : 100
2026-07-26 00:42:12 | INFO |   Delay between batches          : 0.5s
2026-07-26 00:42:12 | INFO |   Estimated publish time         : ~50s


In [33]:
# =============================================================================
# 7. CONVERT ROWS TO JSON MESSAGES
# =============================================================================

log.info("Converting records to JSON messages...")

df_json = df_stream.withColumn(
    "value",
    F.to_json(F.struct([F.col(c) for c in STREAM_COLUMNS]))
).withColumn(
    "key",
    F.col("id").cast(StringType())
).select("key", "value")

log.info("  JSON conversion complete.")


2026-07-26 00:42:12 | INFO | Converting records to JSON messages...
2026-07-26 00:42:12 | INFO |   JSON conversion complete.


In [34]:
# =============================================================================
# 8. PUBLISH TO KAFKA USING toLocalIterator()
#
# toLocalIterator() streams partitions to the driver one at a time,
# avoiding loading the full dataset into memory the way collect() would.
# This is more memory-efficient for large datasets.
#
# Note: The artificial per-batch delay (PUBLISH_DELAY_SEC) is intentional
# — it simulates real-time vehicle listing arrival for streaming analytics.
# Without the delay, all records would publish instantly, which doesn't
# represent live marketplace behavior. This is why we use toLocalIterator()
# rather than Spark's native Kafka sink (which has no delay mechanism).
# =============================================================================

log.info(f"Publishing records to Kafka topic: {KAFKA_TOPIC}...")

publish_start     = time.time()
records_published = 0
batch_number      = 0
failed_batches    = 0
current_batch     = []

total_batches = (records_to_send + BATCH_SIZE - 1) // BATCH_SIZE

for row in df_json.toLocalIterator():
    current_batch.append(row)

    if len(current_batch) == BATCH_SIZE:
        batch_number += 1
        try:
            batch_df = spark.createDataFrame(current_batch, df_json.schema)
            batch_df.write \
                .format("kafka") \
                .option("kafka.bootstrap.servers", KAFKA_BROKER) \
                .option("topic", KAFKA_TOPIC) \
                .save()

            records_published += len(current_batch)
            current_batch      = []

            if batch_number % 10 == 0 or batch_number == total_batches:
                elapsed    = round(time.time() - publish_start, 1)
                throughput = round(
                    records_published / elapsed, 1
                ) if elapsed > 0 else 0
                log.info(
                    f"  Batch {batch_number:>4}/{total_batches} | "
                    f"Published: {records_published:>6} | "
                    f"Elapsed: {elapsed:>6}s | "
                    f"Throughput: {throughput} rec/s"
                )

            time.sleep(PUBLISH_DELAY_SEC)

        except Exception as e:
            failed_batches += 1
            log.error(f"  ✗ Batch {batch_number} failed: {e}")
            current_batch = []
            continue

# Flush any remaining records that didn't fill a complete batch
if current_batch:
    batch_number += 1
    try:
        batch_df = spark.createDataFrame(current_batch, df_json.schema)
        batch_df.write \
            .format("kafka") \
            .option("kafka.bootstrap.servers", KAFKA_BROKER) \
            .option("topic", KAFKA_TOPIC) \
            .save()
        records_published += len(current_batch)
        log.info(
            f"  Batch {batch_number:>4}/{total_batches} | "
            f"Flushed remaining: {len(current_batch)} records"
        )
    except Exception as e:
        failed_batches += 1
        log.error(f"  ✗ Flush batch failed: {e}")

publish_end  = time.time()
publish_time = round(publish_end - publish_start, 2)

2026-07-26 00:42:12 | INFO | Publishing records to Kafka topic: vehicle_market_stream...
2026-07-26 00:42:18 | INFO |   Batch   10/100 | Published:   1000 | Elapsed:    5.9s | Throughput: 169.5 rec/s
2026-07-26 00:42:23 | INFO |   Batch   20/100 | Published:   2000 | Elapsed:   11.5s | Throughput: 173.9 rec/s
2026-07-26 00:42:29 | INFO |   Batch   30/100 | Published:   3000 | Elapsed:   16.9s | Throughput: 177.5 rec/s
2026-07-26 00:42:34 | INFO |   Batch   40/100 | Published:   4000 | Elapsed:   22.6s | Throughput: 177.0 rec/s
2026-07-26 00:42:40 | INFO |   Batch   50/100 | Published:   5000 | Elapsed:   28.4s | Throughput: 176.1 rec/s
2026-07-26 00:42:46 | INFO |   Batch   60/100 | Published:   6000 | Elapsed:   34.2s | Throughput: 175.4 rec/s
2026-07-26 00:42:52 | INFO |   Batch   70/100 | Published:   7000 | Elapsed:   40.1s | Throughput: 174.6 rec/s
2026-07-26 00:42:58 | INFO |   Batch   80/100 | Published:   8000 | Elapsed:   45.9s | Throughput: 174.3 rec/s
2026-07-26 00:43:03 | I

In [35]:
# =============================================================================
# 9. VERIFY MESSAGES IN TOPIC
# =============================================================================

log.info("Verifying published messages by reading from Kafka...")

try:
    df_verify = spark.read \
        .format("kafka") \
        .option("kafka.bootstrap.servers", KAFKA_BROKER) \
        .option("subscribe", KAFKA_TOPIC) \
        .option("startingOffsets", "earliest") \
        .option("endingOffsets", "latest") \
        .load()

    messages_in_topic = df_verify.count()
    log.info(f"  Messages confirmed in topic : {messages_in_topic}")

    # Validate message count — catch partial writes
    if messages_in_topic < records_published:
        raise ValueError(
            f"Kafka contains {messages_in_topic} messages but "
            f"{records_published} were published. "
            f"Possible partial write — check broker logs."
        )
    log.info("  Message count validation passed.")

    # Sample and display raw messages
    df_sample = df_verify \
        .select(F.col("value").cast(StringType())) \
        .limit(3)
    log.info("  Sample messages (first 3):")
    df_sample.show(3, truncate=80)

    # Parse one message back to JSON to confirm consumer deserialization
    sample_raw = df_sample.first()[0]
    parsed     = json.loads(sample_raw)
    log.info(f"  JSON parse verification — keys: {list(parsed.keys())}")
    log.info("  JSON parsing verification passed.")

    verification_status = "✓ PASSED"

except ValueError as ve:
    messages_in_topic   = messages_in_topic if 'messages_in_topic' in dir() else 0
    verification_status = f"✗ FAILED: {ve}"
    log.error(f"  Verification failed: {ve}")

except Exception as e:
    messages_in_topic   = "Unknown"
    verification_status = f"✗ FAILED: {e}"
    log.error(f"  Verification failed: {e}")



2026-07-26 00:43:10 | INFO | Verifying published messages by reading from Kafka...
26/07/26 00:43:10 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
2026-07-26 00:43:11 | INFO |   Messages confirmed in topic : 40000              
2026-07-26 00:43:11 | INFO |   Message count validation passed.
2026-07-26 00:43:11 | INFO |   Sample messages (first 3):
26/07/26 00:43:11 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/07/26 00:43:11 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
2026-07-26 00:43:11 | INFO |   JSON parse verification — keys: ['id', 'manufacturer', 'model', 'year', 'price', 'odometer',

+--------------------------------------------------------------------------------+
|                                                                           value|
+--------------------------------------------------------------------------------+
|{"id":"7302062311","manufacturer":"lexus","model":"ct 200h","year":2016,"pric...|
|{"id":"7312588334","manufacturer":"toyota","model":"corolla","year":2006,"pri...|
|{"id":"7310127460","manufacturer":"unknown","model":"freightliner flc120","ye...|
+--------------------------------------------------------------------------------+



In [36]:
# =============================================================================
# 10. PRODUCER VALIDATION REPORT
# =============================================================================

pipeline_end   = time.time()
execution_time = round(pipeline_end - pipeline_start, 2)
avg_throughput = round(
    records_published / publish_time, 1
) if publish_time > 0 else 0

publish_status = "✓ SUCCESS" if failed_batches == 0 \
    else f"⚠ PARTIAL ({failed_batches} batches failed)"

log.info("\n" + "="*65)
log.info("      KAFKA PRODUCER — VALIDATION REPORT")
log.info("="*65)
log.info(f"  Execution Time          : {execution_time}s")
log.info(f"  Publish Time            : {publish_time}s")
log.info(f"  Kafka Broker            : {KAFKA_BROKER}")
log.info(f"  Kafka Topic             : {KAFKA_TOPIC}")
log.info(f"  Topic Status            : {topic_status}")
log.info(f"  Topic Partitions        : 3")
log.info(f"  Source Dataset          : Silver Featured")
log.info(f"  Records Available       : {total_available}")
log.info(f"  Records Selected        : {records_to_send}")
log.info(f"  Records Published       : {records_published}")
log.info(f"  Messages in Topic       : {messages_in_topic}")
log.info(f"  Failed Batches          : {failed_batches}")
log.info(f"  Batch Size              : {BATCH_SIZE}")
log.info(f"  Publish Delay           : {PUBLISH_DELAY_SEC}s")
log.info(f"  Avg Throughput          : {avg_throughput} records/sec")
log.info(f"  Iterator Strategy       : toLocalIterator()")
log.info(f"  Verification            : {verification_status}")
log.info(f"  Publish Status          : {publish_status}")
log.info("")

if failed_batches == 0 and verification_status.startswith("✓"):
    log.info("  ✓ Kafka Producer complete.")
    log.info("  ✓ Vehicle records successfully streamed to Kafka.")
    log.info("  ✓ Ready for Spark Structured Streaming consumer.")
else:
    log.info("  ⚠ Producer completed with issues. Review errors above.")

log.info("="*65)

2026-07-26 00:43:11 | INFO | 
2026-07-26 00:43:11 | INFO |       KAFKA PRODUCER — VALIDATION REPORT
2026-07-26 00:43:11 | INFO | =================================================================
2026-07-26 00:43:11 | INFO |   Execution Time          : 60.49s
2026-07-26 00:43:11 | INFO |   Publish Time            : 58.04s
2026-07-26 00:43:11 | INFO |   Kafka Broker            : localhost:9092
2026-07-26 00:43:11 | INFO |   Kafka Topic             : vehicle_market_stream
2026-07-26 00:43:11 | INFO |   Topic Status            : existing
2026-07-26 00:43:11 | INFO |   Topic Partitions        : 3
2026-07-26 00:43:11 | INFO |   Source Dataset          : Silver Featured
2026-07-26 00:43:11 | INFO |   Records Available       : 242979
2026-07-26 00:43:11 | INFO |   Records Selected        : 10000
2026-07-26 00:43:11 | INFO |   Records Published       : 10000
2026-07-26 00:43:11 | INFO |   Messages in Topic       : 40000
2026-07-26 00:43:11 | INFO |   Failed Batches          : 0
2026-07-26 00:43